In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO02 - Business Travel
   
   LOGIC SUMMARY:
   1. MASTER AU LIST: Uses fy25_list_of_unit to guarantee all AUs appear in output.
   2. HIGH RISK JURISDICTIONS: Queries td_country_risk_rating_abac for 'High' risk.
   3. TRAVEL DATA UNION: 
      - AMEX: Extracts Cost Center, Country, and TRY_CASTs Numberoftrips to INT.
      - CWT: Extracts Cost Center, Country, and hardcodes '1' as the trip count 
        since the CWT schema lacks a Numberoftrips column.
   4. FILTERING: INNER JOINS Travel Data to the High Risk Countries list.
   5. MAPPING & AGGREGATING: Joins high-risk trips to the Cost Center Mapping Bootstrap, 
      then SUMS the total number of trips per AU_ID.
   6. FINAL OUTPUT: LEFT JOINS aggregated trip sums back to the Master AU List.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    -- AMEX Data (Has Numberoftrips)
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    -- CWT Data (No Numberoftrips, counting each row as 1)
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    SELECT 
        t.Cleaned_CC,
        t.DestinationCountry,
        t.Trip_Count,
        t.Source_System
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

Trips_By_AU AS (
    SELECT 
        m.AU_ID,
        SUM(h.Trip_Count) AS Total_Trips
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN High_Risk_Trips h
        ON CAST(m.Cost_Center_ID AS INT) = h.Cleaned_CC
    GROUP BY m.AU_ID
)

SELECT 
    u.BusinessID,                          
    u.AU_Name,                             
    'GEO02' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    COALESCE(CAST(t.Total_Trips AS STRING), '0') AS Response
    
FROM Master_AUs u
LEFT JOIN Trips_By_AU t 
    ON u.BusinessID = t.AU_ID
ORDER BY u.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO02 - Business Travel Drill-Down
=================================================================================== */

WITH High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'Original/Unknown' AS CWT_Transaction_Type,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        TRIM(TransactionType) AS CWT_Transaction_Type,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    t.Source_System,
    t.DestinationCountry AS High_Risk_Destination,
    t.Trip_Count,
    t.CWT_Transaction_Type
    
FROM vw_cost_center_mapping_bootstrap m

INNER JOIN Combined_Travel_Data t
    ON CAST(m.Cost_Center_ID AS INT) = t.Cleaned_CC
    
INNER JOIN High_Risk_Countries h
    ON t.DestinationCountry = h.CountryName
    
ORDER BY BusinessID;